# Notebook 02 — Randomized Hyperparameter Search

**Session:** Hyper-parameter optimization  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Sample continuous hyperparameter distributions (`scipy.stats.loguniform`) using
`RandomizedSearchCV`. Unlike Grid Search, Random Search explores distinct values along
every hyperparameter axis on each draw, making it dramatically more sample-efficient
when only a few hyperparameters strongly govern validation performance.

In [ ]:
import time

from scipy.stats import loguniform
from sklearn import model_selection, svm
from sklearn.datasets import make_classification

## 1. Generate Synthetic Classification Dataset

In [ ]:
X, y = make_classification(
    n_samples=300,
    n_features=50,
    n_informative=10,
    n_redundant=25,
    n_repeated=15,
    n_clusters_per_class=5,
    flip_y=0.05,
    class_sep=0.5,
    random_state=0,
)
print(f"Dataset shape: X={X.shape}, y={y.shape}")

## 2. Continuous Log-Uniform Distributions & RandomizedSearchCV

In [ ]:
svc = svm.SVC(probability=True, random_state=1)

search_distributions = {
    "kernel": ["linear", "rbf"],
    "C": loguniform(1, 1000),
    "gamma": loguniform(0.0001, 0.1),
}

search_func = model_selection.RandomizedSearchCV(
    estimator=svc,
    param_distributions=search_distributions,
    n_iter=10,
    scoring="accuracy",
    n_jobs=-1,
    cv=5,
    random_state=0,
)

start = time.perf_counter()
search_func.fit(X, y)
elapsed = time.perf_counter() - start

print(f"RandomizedSearchCV finished in {elapsed:.2f}s")
print("Best params:", search_func.best_params_)
print(f"Best 5-fold CV accuracy: {search_func.best_score_:.4f}")